# Practical PyTorch · Part 19 — Companion Notebook

Embeddings: turn inputs into vectors you can search and compare. If classification is 'which bucket?', embeddings are 'what is this close to?'

In [ ]:
!pip install -q sentence-transformers

## Text to vectors

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

texts = [
    'How do I reset my password?',
    'I forgot my login and need a new password.',
    'The image classifier thinks my dog is a wolf.',
]
emb = model.encode(texts, convert_to_tensor=True)
print(emb.shape)   # torch.Size([3, 384]) — one 384-dim vector per text

## Compare meaning with cosine similarity

In [ ]:
from sentence_transformers import util

scores = util.cos_sim(emb[0], emb)
print(scores)
# tensor([[1.00, 0.62, 0.09]]) — identical to itself, close to sentence 2, far from sentence 3
# higher = more alike in meaning (1.0 same direction, 0 unrelated)

## A tiny semantic search

In [ ]:
import torch

docs = [
    'Reset your password from the account settings page.',
    'Use a GPU runtime when image generation is too slow.',
    'Model cards usually include license and intended-use details.',
]
query = 'Where do I find the license for a model?'

doc_emb = model.encode(docs, convert_to_tensor=True)
q_emb = model.encode(query, convert_to_tensor=True)

scores = util.cos_sim(q_emb, doc_emb)[0]
top = torch.topk(scores, k=len(docs))        # a *ranked* list, not just the best
for score, idx in zip(top.values, top.indices):
    print(f'{score:.2f}  {docs[idx]}')
# the model-card sentence ranks first, despite sharing no keywords with the query

## This is the retrieval in RAG

That search *is* the **R** in RAG (retrieval-augmented generation): embed the question, find the nearest documents, and paste them into a language model's prompt as context. At scale you swap the Python list for a **vector database** (FAISS, Chroma, pgvector) — same idea, faster nearest-neighbour search.

Embeddings compare meaning, not string overlap.

**Next: Part 20 — Build a Semantic Search Engine.**